In [ ]:
# ===============================================
# IRP SLOT CHECKER WITH TELEGRAM NOTIFICATIONS
# ===============================================
# This script automatically checks for available IRP appointment slots
# and sends notifications via Telegram when slots are found

import time            # For sleep delays
import schedule        # For scheduling jobs at specific times
import requests        # For making HTTP requests to Telegram API
from datetime import datetime, timedelta, time as dtime  # For date/time handling
from selenium import webdriver                           # For browser automation
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait  # For waiting for elements
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options    # For Chrome browser settings
from bs4 import BeautifulSoup                           # For parsing HTML content
import logging                                           # For detailed logging/debugging

# Setup logging to track script activity and debug issues
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# ==========================================
# TELEGRAM BOT CONFIGURATION
# ==========================================
# Replace these with your actual Telegram bot token and chat ID
# Get bot token from @BotFather on Telegram
# Get chat ID by messaging your bot and checking the API
TELEGRAM_TOKEN = ""  # Your Telegram bot token
CHAT_ID = ""           # Your Telegram chat ID

def send_telegram(message):
    """
    Sends a message to Telegram using the bot API
    
    Args:
        message (str): The message text to send
    
    Returns:
        None: Logs success/failure of message sending
    """
    try:
        # Construct the Telegram API URL for sending messages
        url = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendMessage"
        
        # Send POST request to Telegram API with chat ID and message
        response = requests.post(url, data={"chat_id": CHAT_ID, "text": message})
        
        # Check if message was sent successfully (HTTP 200 = success)
        if response.status_code == 200:
            logging.info("✅ Telegram message sent successfully")
        else:
            logging.error(f"❌ Failed to send Telegram message: {response.status_code}")
    except Exception as e:
        # Log any errors that occur during message sending
        logging.error(f"❌ Telegram error: {e}")



# ==========================================
# SELENIUM BROWSER SETUP
# ==========================================
def setup_driver():
    """
    OPTION 1: Connect to your existing Chrome session (RECOMMENDED)
    
    SETUP STEPS BEFORE RUNNING THIS SCRIPT:
    1. Close ALL Chrome instances first
    2. Open Command Prompt/Terminal
    3. Start Chrome with remote debugging:
       
       Windows: 
       "C:\Program Files\Google\Chrome\Application\chrome.exe" --remote-debugging-port=9222 --user-data-dir="C:\selenium\ChromeProfile"
       
       Mac: 
       /Applications/Google\ Chrome.app/Contents/MacOS/Google\ Chrome --remote-debugging-port=9222 --user-data-dir="/tmp/selenium/ChromeProfile"
       
       Linux:
       google-chrome --remote-debugging-port=9222 --user-data-dir="/tmp/selenium/ChromeProfile"
    
    4. Log into the IRP website in this Chrome instance
    5. Navigate to the appointment page
    6. Then run this script
    
    Returns:
        webdriver.Chrome: Connected to your existing Chrome session
    """
    # Configure Chrome to connect to existing session
    chrome_options = Options()
    chrome_options.add_experimental_option("debuggerAddress", "127.0.0.1:9222")
    
    try:
        # Connect to the existing Chrome session
        driver = webdriver.Chrome(options=chrome_options)
        logging.info("✅ Connected to existing Chrome session successfully")
        return driver
    except Exception as e:
        logging.error(f"❌ Failed to connect to existing Chrome session: {e}")
        logging.error("🔧 Make sure Chrome is running with --remote-debugging-port=9222")
        raise

def setup_driver_new_session():
    """
    OPTION 2: Create new Chrome session (requires manual login)
    Use this if Option 1 doesn't work for you
    
    Returns:
        webdriver.Chrome: New Chrome driver instance
    """
    chrome_options = Options()
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    # Uncomment for headless mode (runs invisibly)
    # chrome_options.add_argument("--headless")
    
    driver = webdriver.Chrome(options=chrome_options)
    driver.maximize_window()
    logging.info("⚠ New Chrome session created - you need to login manually")
    return driver

# ==========================================
# CHOOSE YOUR SETUP METHOD:
# ==========================================
# OPTION 1: Connect to existing logged-in Chrome (RECOMMENDED)
driver = setup_driver()

# OPTION 2: If Option 1 fails, uncomment the line below and comment the line above
# driver = setup_driver_new_session()

# ==========================================
# HTML PARSING FOR SLOT EXTRACTION (MULTI-MONTH)
# ==========================================
def parse_slots_multi_month(page_html):
    """
    Extracts available appointment slots for September, October, November, December
    by navigating the calendar using the 'next' button.
    
    Args:
        driver (webdriver.Chrome): Active Selenium WebDriver
    
    Returns:
        list: List of available slots in format "Day Month Year (YYYY-MM-DD)"
    """
    slots = []  # Store all available slots

    try:
        # Loop through current + next 3 months (total 4 months)
        for _ in range(2):
            # Parse current month's calendar
            soup = BeautifulSoup(page_html, "html.parser")
            calendar_table = soup.find("table", class_="table-condensed")
                
            if calendar_table:
                for td in calendar_table.find_all("td", attrs={"data-date": True}):
                    classes = td.get("class", [])

                    # ✅ Available days have class "day" but NOT "disabled"
                    if "day" in classes and "disabled" not in classes:
                        print("Slots are available");
                        try:
                            # Extract timestamp from data-date attribute (in milliseconds)
                            timestamp = int(td["data-date"]) // 1000  # Convert to seconds
                    
                            # Convert timestamp to datetime object
                            date_obj = datetime.utcfromtimestamp(timestamp)
                            month = date_obj.strftime("%B")  # Get full month name

                            # Only include Sept → Dec
                            if month in ["September", "October", "November", "December"]:
                                day_num = td.text.strip() # Get the day number from cell text
                                date_str = date_obj.strftime("%Y-%m-%d") # Format as YYYY-MM-DD
                                slots.append(f"{day_num} {month} {date_obj.year} ({date_str})")

                        except Exception as e:
                            logging.error(f"Error parsing date from td: {e}")
                            continue

            # ✅ Click the "next" button to go to next month
            try:
                next_btn = driver.find_element(By.CLASS_NAME, "next")
                driver.execute_script("arguments[0].click();", next_btn)  # safer than .click()
                time.sleep(2)  # wait for calendar refresh
            except Exception as e:
                logging.warning(f"⚠ Could not click next button on calendar: {e}")
                break

    except Exception as e:
        logging.error(f"Error parsing slots: {e}")
    #return only slots for original logic
    #trimmedSlots = slots[:10]
    return slots

# ==========================================
# MAIN SLOT CHECKING LOGIC
# ==========================================
def check_irp_slots():
    logging.info(f"⏳ Checking slots at {datetime.now().strftime('%H:%M:%S')}...")
    try:
        # STEP 1: Navigate or refresh
        target_url = "https://portal.irishimmigration.ie/en/reschedule_appointment/"
        current_url = driver.current_url

        if target_url not in current_url:
            logging.info("🔄 Navigating to appointment page...")
            driver.get(target_url)
        else:
            logging.info("🔄 Refreshing current page...")
            driver.refresh()

        # STEP 2: Wait for page load
        WebDriverWait(driver, 40).until(
            lambda d: d.execute_script("return document.readyState") == "complete"
        )
        time.sleep(1)

        # STEP 3: If still on Continue page, try to click through
        if "Continue" in driver.page_source:
            continue_selectors = [
                "//button[contains(@onclick,'continueHandler')]",
                "//button[@class='primary-button']",
                "//button[contains(translate(text(), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'continue')]",
                "//button[contains(text(),'Continue')]",
                "//input[@type='button' and contains(@value,'Continue')]",
                "//*[contains(translate(text(), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'continue') and (@onclick or @type='button' or local-name()='button')]"
            ]

            try:
                continue_found = False
                for i, selector in enumerate(continue_selectors, 1):
                    try:
                        logging.info(f"🔍 Trying Continue button strategy {i}...")
                        continue_btn = WebDriverWait(driver, 10).until(
                            EC.element_to_be_clickable((By.XPATH, selector))
                        )
                        driver.execute_script("arguments[0].click();", continue_btn)
                        logging.info("✅ Clicked Continue button")
                        continue_found = True
                        break
                    except:
                        continue

                if not continue_found:
                    logging.warning("⚠ No Continue button found, may already be on calendar page")

            except Exception as e:
                logging.warning(f"⚠ Continue click handling failed: {e}")

        # STEP 4: Explicitly wait for calendar to appear
        try:
            WebDriverWait(driver, 50).until(
                EC.presence_of_element_located((By.XPATH, "//td[@data-date]"))
            )
            logging.info("✅ Calendar loaded successfully")
        except:
            logging.error("❌ Still stuck on Continue page, calendar not visible")
            return False

        time.sleep(2)

        # STEP 5: Extract slots
        slots = parse_slots_multi_month(driver.page_source)

        # STEP 6: Send results
        if slots:
            msg = f"🎯 IRP Slots Found ({len(slots)} available):\n\n" + "\n".join([f"📅 {slot}" for slot in slots])
            logging.info(f"✅ Found {len(slots)} available slots")
            send_telegram(msg)
            return True
        else:
            time_str = datetime.now()
            logging.info(f"❌ No slots available at : {time_str}")
            return False

    except Exception as e:
        logging.error(f"❌ Error in check_irp_slots: {e}")
        return False

# ==========================================
# RETRY LOGIC AND SCHEDULING FUNCTIONS
# ==========================================
"""
    Wrapper function that implements retry logic:
    - First attempt to check slots
    - If no slots found, wait 5 minutes and try once more
    - This prevents false negatives due to temporary loading issues
"""
def check_with_retry(interval=300):
    #success = 
    check_irp_slots()  # First attempt
    
    # if not success:
    #     # If first attempt failed or found no slots, wait and retry
    #     logging.info("⏳ Retrying in 5 minutes...")
    #     time.sleep(interval)  # Sleep for 5 minutes (300 seconds)
    #     check_irp_slots()  # Second attempt (no infinite retry to avoid spam)

def schedule_window(start: dtime, end: dtime, interval=5):
    """
    Safely schedule jobs between two times (supports midnight crossing).
    
    Args:
        start (dtime): Start time (e.g., dtime(9, 55))
        end (dtime): End time (e.g., dtime(10, 15))
        interval (int): Interval in minutes
    """
    today = datetime.today()

    # Anchor start and end to today
    current = datetime.combine(today, start)
    end_dt = datetime.combine(today, end)

    # If end is "before" start, it means it crosses midnight → push end to next day
    if end < start:
        end_dt += timedelta(days=1)

    while current <= end_dt:
        time_str = current.strftime("%H:%M")
        schedule.every().day.at(time_str).do(check_with_retry)
        logging.info(f"⏰ Scheduled job at {time_str}")
        current += timedelta(minutes=interval)


# ==========================================
# SCHEDULING CONFIGURATION
# ==========================================
# Clear any existing scheduled jobs (useful when re-running in Jupyter)
schedule.clear()

# SCHEDULE your timings according to your need. 
# This creates jobs at: 9:55, 10:00, 10:05, 10:10, 10:15
# Peak time when new slots might be released
schedule_window(dtime(7, 18), dtime(12, 10), interval=2)
# schedule_window(dtime(23, 0), dtime(2, 0), interval=10)
# schedule_window(dtime(18, 14), dtime(18, 45), interval=1)

# SCHEDULE 2: Check at specific exact times throughout the day
# These times are strategic - likely when slots are released or updated
fixed_times = [
    "12:30",  # Lunch time check 
]

# Add each fixed time as a scheduled job
for t in fixed_times:
    schedule.every().day.at(t).do(check_with_retry)

# ==========================================
# DISPLAY SCHEDULED JOBS AND START MONITORING
# ==========================================
# Show all scheduled jobs for verification
logging.info("📅 Scheduled times:")


logging.info("🚀 IRP slot checker is running. Waiting for trigger times...")
logging.info("💡 Press Ctrl+C to stop the script")

# ==========================================
# MAIN EXECUTION LOOP
# ==========================================
# This loop runs continuously, checking if any scheduled jobs need to run
try:
    while True:
        # Check if any scheduled jobs are due to run
        schedule.run_pending()
        
        # Sleep for 30 seconds before checking again
        # (30 seconds provides good responsiveness without excessive CPU usage)
        time.sleep(1)
        
except KeyboardInterrupt:
    # Handle Ctrl+C gracefully
    logging.info("🛑 Script stopped by user")
    driver.quit()  # Close the browser
    
except Exception as e:
    # Handle any unexpected errors
    logging.error(f"❌ Unexpected error: {e}")
    driver.quit()  # Close the browser

# ==========================================
# USAGE INSTRUCTIONS:
# ==========================================


"""
HOW TO USE THIS SCRIPT:

1. SETUP:
   - Replace TELEGRAM_TOKEN with your bot token from @BotFather
   - Replace CHAT_ID with your chat ID
   - Make sure Chrome browser is installed
   - Install required packages: pip install selenium beautifulsoup4 schedule requests

2. BEFORE RUNNING:
   - Manually log into the IRP website in Chrome
   - Navigate to the appointment scheduling page
   - Keep that Chrome window open

3. RUNNING:
   - Run this script in the same environment
   - It will use the existing logged-in session0
   - The script will run continuously, checking at scheduled times

4. MONITORING:
   - Watch the console for log messages
   - You'll get Telegram notifications when slots are found
   - Press Ctrl+C to stop the script

5. TROUBLESHOOTING:
   - If calendar doesn't load, check the website structure
   - Adjust the XPath selectors if needed  
   - Test manually with check_irp_slots() function first
   """